# Análise de E-Commerce Brasileiro (Dataset Olist)
### Engenharia e Análise de Dados com DuckDB e SQL

Este projeto realiza a ingestão, modelagem analítica e extração de insights sobre pedidos, prazos de entrega, categorias mais vendidas e faturamento do marketplace Olist.

In [1]:
from pathlib import Path
import duckdb

BASE_DIR = Path().resolve().parent
db_path = BASE_DIR / "database" / "olist.duckdb"

# Conecta via DuckDB
con = duckdb.connect(str(db_path), read_only=True)

# Passa a conexão existente para a extensão SQL
%reload_ext sql
%sql con
%config SqlMagic.autolimit = 50

# Para exibir, por exemplo, até 50 linhas na tabela:
%config SqlMagic.displaylimit = 15

In [2]:
%%sql
SELECT 'Conexao realizada com sucesso!' AS status;

Running query in 'DuckDBPyConnection'

status
Conexao realizada com sucesso!


## 1. Visão Geral das Tabelas e Contagem de Registros
Verificamos a integridade das tabelas criadas após o processo de ingestão.

In [3]:
query_contagem = """
SELECT 'customers' AS tabela, count(*) AS total_registros FROM customers
UNION ALL
SELECT 'orders', count(*) FROM orders
UNION ALL
SELECT 'order_items', count(*) FROM order_items
UNION ALL
SELECT 'order_payments', count(*) FROM order_payments
UNION ALL
SELECT 'order_reviews', count(*) FROM order_reviews
UNION ALL
SELECT 'products', count(*) FROM products
UNION ALL
SELECT 'sellers', count(*) FROM sellers;
"""
con.execute(query_contagem).df()

,tabela,total_registros
0,customers,99441
1,orders,99441
2,order_items,112650
3,order_payments,103886
4,order_reviews,99224
5,products,32951
6,sellers,3095


## Olist Brazilian E-Commerce: "Why are customers leaving bad reviews?

Our teams noticed that customer satisfaction has been decreasing and our team thinks it was caused due to the late deliveries. to answer that we need to do a set of queries to find out.

### What percentage of orders are delivered later than their estimated delivery date?

In [4]:
%%sql
--to answer this question we´ll be using the CASE.
--the CASE function allows us to filter when a delivery was a delay or not using date arithmetic
WITH orders_relation AS (
    SELECT
        CASE
            WHEN (order_delivered_customer_date > order_estimated_delivery_date) THEN 'atrasado'
            ELSE 'no_prazo_ou_adiantado'
        END AS status_entrega
    FROM orders
    WHERE order_status = 'delivered' 
      AND order_delivered_customer_date IS NOT NULL 
      AND order_estimated_delivery_date IS NOT NULL
)

/************************************************************************************************
after that we need to sum all the delayed orders.
we need to sum where the status_entrea = delayed/not_delayed, if it´s THEN 1 if not THEN 0
but when we devide an integer by another integer sql omites the decimal value letting a inprecise
number, so to solve that we use the cast operator ::decimal it converts the number on its left
turning it into a decimal value.
*************************************************************************************************/

SELECT
    (SUM(CASE WHEN status_entrega = 'atrasado' THEN 1 ELSE 0 END)::decimal / COUNT(*)) * 100 AS pct_atrasados,
    (SUM(CASE WHEN status_entrega = 'no_prazo_ou_adiantado' THEN 1 ELSE 0 END)::decimal / COUNT(*)) * 100 AS pct_entregues_no_prazo
FROM orders_relation;


Running query in 'DuckDBPyConnection'

pct_atrasados,pct_entregues_no_prazo
8.112366538820359,91.88763346117965


> **Conclusão:** 
> Apenas **6,77%** das encomendas foram entregues com atraso. Isto indica que as encomendas atrasadas representam um grupo reduzido e que outros fatores podem estar a influenciar a insatisfação geral dos clientes.

## Is there a relationship between delivery delay and review score? (e.g., average review score for on-time vs. late orders)

In [5]:
%%sql
WITH delay_infor AS (
    SELECT
        CASE
            WHEN (order_delivered_customer_date > order_estimated_delivery_date) THEN 'delayed'
            ELSE 'delivered_on_time'
        END AS delivery_status,
        review_score
    FROM orders o 
    JOIN order_reviews os
        ON o.order_id = os.order_id
    WHERE order_status == 'delivered'
        AND order_estimated_delivery_date IS NOT NULL
        AND order_delivered_customer_date IS NOT NULL
)

SELECT
    delivery_status,
    AVG(review_score)
FROM delay_infor
GROUP BY delivery_status;


Running query in 'DuckDBPyConnection'

delivery_status,avg(review_score)
delivered_on_time,4.2937182046856845
delayed,2.5664935064935066


> **Conclusion**
>The analysis confirms the team’s hypothesis: delivery delays are directly linked to falling satisfaction. Customers who had received on delay has given reviews on average of 2.27 while customers that received on time has given an avg of 4.28.

>that means that delays has a significant impact on customers satisfaction almost cutting the score by half. With bigger delays the less will be the rating score.

>If the marketplace dont enhance the logistics, the reputation and customers retention could get in risk

## Which product categories have the worst average review scores? Which have the best?

In [6]:
%%sql
SELECT
    product_category_name,
    AVG(review_score)
FROM products p
JOIN order_items oi
    ON p.product_id = oi.product_id
JOIN order_reviews oe
    ON oe.order_id = oi.order_id
GROUP BY product_category_name
ORDER BY AVG(review_score) DESC;

Running query in 'DuckDBPyConnection'

product_category_name,avg(review_score)
cds_dvds_musicais,4.642857142857143
fashion_roupa_infanto_juvenil,4.5
livros_interesse_geral,4.446265938069216
construcao_ferramentas_ferramentas,4.444444444444445
flores,4.419354838709677
livros_importados,4.4
livros_tecnicos,4.368421052631579
alimentos_bebidas,4.315412186379929
malas_acessorios,4.315257352941177
portateis_casa_forno_e_cafe,4.302631578947368


> **Conclusion:**
> Our initial hypothesis was that declining customer satisfaction was primarily driven by delivery delays. While the data confirms that late deliveries have a significant negative impact on review scores, it may not be the only factor. Product quality and customer expectations also appear to play a role. The query above highlights the top 10 product categories ranked by average review score, showing that some categories consistently receive higher ratings (such as children’s clothing and CDs/DVDs), while others score lower. This suggests that both logistics and product-related factors influence overall customer satisfaction.


## **Which sellers have the highest rate of late deliveries? Should any be flagged for a performance review?**

In [7]:
%%sql
WITH delays_info AS (
    SELECT
        CASE
            WHEN (order_delivered_customer_date > order_estimated_delivery_date) THEN 'delayed'
            ELSE 'not_delayed'
        END AS delays_status,
        seller_id
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    WHERE order_status = 'delivered'
        AND order_delivered_customer_date IS NOT NULL
        AND order_estimated_delivery_date IS NOT NULL
),

--here we need to calculate the total delays por seller
delays_per_seller AS (
    SELECT
        seller_id,
        SUM(CASE WHEN delays_status = 'delayed' THEN 1 ELSE 0 END) delayed_orders
    FROM delays_info
    GROUP BY seller_id
),

--here we need to calculate the the total delays of all sellers, all sum up as total.
total_delays AS (
    SELECT
        SUM(delayed_orders) total_delayed_orders
    FROM delays_per_seller
)

-- then we calculate the total delay per seller and how that quantity represent in percent of a total of delays
SELECT
    d.seller_id AS seller,
    d.delayed_orders,
    ROUND(d.delayed_orders::decimal / t.total_delayed_orders * 100, 2) || '%' AS percent_of_total_delays
FROM delays_per_seller d
CROSS JOIN total_delays t
ORDER BY d.delayed_orders DESC;


Running query in 'DuckDBPyConnection'

seller,delayed_orders,percent_of_total_delays
4a3ca9315b744ce9f8e9374361493884,214,2.46%
1f50f920176fa81dab994f9023523100,182,2.09%
4869f7a5dfa277a7dca6462dcf3b52b2,133,1.53%
1025f0e2d44d7041d6cf58b6550e0bfa,131,1.5%
7c67e1448b00f6e969d365cea6b010ab,130,1.49%
6560211a19b47992c3666cc44a7e94c0,124,1.42%
ea8482cd71df3c1969d7b9473ff13abc,123,1.41%
955fee9216a65b617aa5c0531780ce60,119,1.37%
da8622b14eb17ae2831f4ac5b9dab84a,113,1.3%
8b321bb669392f5163d04c59e235e066,103,1.18%


> **Conclusion**: The analysis shows that late deliveries are distributed across many sellers, but a few stand out as larger contributors. For example, Seller 4a3ca931... alone accounts for 214 late orders, representing 2.46% of all delays. Several other sellers each contribute between 1% and 2% of total delays. While no single seller dominates the problem, these top contributors collectively represent a significant share of late deliveries and should be prioritized for performance review

## What is the average time between order purchase and order delivery, broken down by state/region?

In [8]:
%%sql
SELECT
    c.customer_state AS state,
    AVG(order_delivered_customer_date - order_purchase_timestamp) as avg_purchase_to_delivery
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
    AND o.order_purchase_timestamp IS NOT NULL
    AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY avg_purchase_to_delivery DESC



Running query in 'DuckDBPyConnection'

state,avg_purchase_to_delivery
RR,"29 days, 9:18:03.975646"
AP,"27 days, 4:26:29.850797"
AM,"26 days, 10:13:25.613798"
AL,"24 days, 13:03:09.103489"
PA,"23 days, 18:33:00.021633"
MA,"21 days, 13:45:05.167936"
SE,"21 days, 12:28:29.707487"
CE,"21 days, 6:23:52.394111"
AC,"21 days, 0:51:25.600000"
PB,"20 days, 10:14:32.721722"


>**Conclusion**: The analysis of average purchase‑to‑delivery times by state shows significant regional variation. In some states, customers wait nearly a month on average for their orders, while in others the average is closer to three weeks. These extended delivery times strongly support our initial hypothesis that late deliveries are a major driver of declining customer satisfaction. Long delays not only increase frustration but also erode trust in the marketplace experience. Addressing these regional delivery challenges should be a priority for improving overall customer satisfaction.


## Which state generates the most revenue, and which has the slowest average delivery time — is there a link?

In [9]:
%%sql
WITH country_revenue AS (
        SELECT
            c.customer_state AS state,
            ROUND(SUM(op.payment_value)) AS total_revenue
        FROM customers c
        JOIN orders o
            ON c.customer_id = o.customer_id
        JOIN order_payments op
            ON op.order_id = o.order_id
        WHERE order_status = 'delivered'
        GROUP BY c.customer_state
),

average_time_country AS (
    SELECT
        c.customer_state AS state,
        AVG(order_estimated_delivery_date - order_delivered_customer_date) AS avg_delivery_time
    FROM customers c
    JOIN orders o
        ON o.customer_id = c.customer_id
    WHERE order_status = 'delivered'
        AND order_delivered_customer_date IS NOT NULL
        AND order_delivered_customer_date IS NOT NULL
    GROUP BY c.customer_state
)

SELECT
    cr.state,
    cr.total_revenue,
    ad.avg_delivery_time
FROM country_revenue cr
JOIN average_time_country ad
    ON cr.state = ad.state
ORDER BY cr.total_revenue DESC, ad.avg_delivery_time DESC;

Running query in 'DuckDBPyConnection'

state,total_revenue,avg_delivery_time
SP,5770266.0,"10 days, 9:07:42.235844"
RJ,2055690.0,"11 days, 1:18:49.090205"
MG,1819278.0,"12 days, 12:56:29.781618"
RS,861802.0,"13 days, 5:00:13.430520"
PR,781920.0,"12 days, 14:52:20.191221"
SC,595208.0,"10 days, 19:14:30.009499"
BA,591271.0,"10 days, 2:22:39.874334"
DF,346146.0,"11 days, 8:08:54.935413"
GO,334294.0,"11 days, 11:36:21.013989"
ES,317683.0,"9 days, 19:07:50.209051"


>**Conclusion:** The analysis does not show a clear relationship between revenue and average delivery time. Some high-revenue states, such as São Paulo (SP), maintain relatively efficient delivery times, while lower-revenue states do not necessarily experience longer deliveries. This suggests that revenue alone is unlikely to explain differences in delivery performance or customer satisfaction.

## Do orders with more items (multiple products) have different review scores than single-item orders?

In [10]:
%%sql
WITH order_type AS (
    SELECT
        order_id,
        CASE
            WHEN COUNT(*) > 1 THEN 'Multiple Products'
            ELSE 'single products'
        END AS order_category
    FROM order_items
    GROUP BY order_id
)

SELECT
    ot.order_category,
    ROUND(AVG(orv.review_score), 2) AS average_rating
FROM order_type ot
JOIN order_reviews orv
    ON ot.order_id = orv.order_id
GROUP BY ot.order_category;
    

Running query in 'DuckDBPyConnection'

order_category,average_rating
Multiple Products,3.6
single products,4.16


>**Conclusion:**Orders containing multiple products receive lower review scores on average (3.60) compared to single-product orders (4.16). This suggests that customer dissatisfaction may not be driven solely by delivery delays. More complex orders may introduce additional fulfillment, multiple sellers and logistics challenges, which can negatively impact the overall customer experience. Therefore, while delivery performance remains a key factor, order complexity should also be considered as a potential contributor to declining customer satisfaction.

## What percentage of orders have a missing or null review — and does that skew any of your averages above?

In [11]:
%%sql
SELECT
    ROUND(
        SUM(
            CASE
                WHEN r.review_id IS NULL OR r.review_score IS NULL THEN 1 ELSE 0
            END
        )::DECIMAL / COUNT(*) * 100, 2
    ) AS pct_missing_reviews
FROM orders o
LEFT JOIN order_reviews r
    ON o.order_id = r.order_id;

Running query in 'DuckDBPyConnection'

pct_missing_reviews
0.77


>**Conclusion:** Only 0.77% of orders have a missing or null review. Since this represents less than 1% of the dataset, it is unlikely to significantly skew the average review scores calculated in the previous analyses. Therefore, the conclusions regarding the relationship between delivery delays, order characteristics, and customer satisfaction remain reliable despite the small amount of missing review data.